# Task 1 — 2019 NYC Yellow Taxi: ingest, validate and prepare

**Student:** fill your own identity in the local config. **Pool:** TR-04 only after checking your Aula allocation. **Question:** Among eligible card trips, predict whether the *recorded* tip exceeds 20% of fare using only pickup-time inputs. Run the ten cells below **in order**, in a fresh kernel on your university Spark cluster. Results and figures remain empty in this template until you run the real 12-month source. Do not paste collision outputs into this notebook.

## 1️⃣ SPARK SESSION CONFIGURATION (ACTUAL RESOURCES)

`make_spark` uses `SparkSession.builder` with the university settings you provide in `config/config.json`; it prints what the live session actually received. A pre-existing SparkContext cannot be resized by editing a notebook cell: restart/re-submit with approved driver/executor resources. Do not assume the reference project's hard-coded 4×6 GB or 400 shuffle partitions are optimal here.

In [ ]:
from pathlib import Path
import sys
from pyspark.sql import SparkSession
ROOT = Path.cwd()
if not (ROOT / "coursework").is_dir():
    ROOT = ROOT.parent  # also works when Jupyter starts inside notebooks/
if not (ROOT / "coursework").is_dir():
    raise RuntimeError("Launch Jupyter from the TR-04 project root or notebooks/ directory")
sys.path.insert(0, str(ROOT))
from coursework.settings import (load_config, make_spark, project_path,
    read_json, require_verified_allocation, spark_configuration)
cfg = load_config()  # gitignored config/config.json: your real allocation and cluster
require_verified_allocation(cfg)  # requires YOUR independently checked Aula row and terms
spark = make_spark(cfg, "Task1")  # config-driven SparkSession.builder, not guessed Colab resources
assert isinstance(spark, SparkSession)
print("Student and allocated dataset:", cfg["student"], cfg["allocation"]["pool_reference"],
      cfg["allocation"]["dataset_name"])
print("ACTUAL Spark application / resources:", spark_configuration(spark))


## 2️⃣ DATA INGESTION + SCHEMA/FILE VALIDATION

Load **all 12 official 2019 TLC Parquet files**, not the UK collision CSV. Spark normalises monthly optional columns/types before `unionByName`. Source byte sizes come from the same Hadoop filesystem that Spark will read; verify driver **and workers** can see the shared POSIX files. The preview is five rows, not a fabricated total. Do not publish row-level data.

In [ ]:
import os
from coursework.data import monthly_paths, hadoop_file_sizes, load_raw_2019
paths = monthly_paths(cfg)
source_files = hadoop_file_sizes(spark, paths)
assert len(paths) == len(source_files) == 12
needed = [paths[0], paths[-1], str(project_path(cfg, "zone_lookup"))]
workers_to_check = min(8, max(1, spark.sparkContext.defaultParallelism))
visible = spark.sparkContext.parallelize(range(workers_to_check), workers_to_check).map(
    lambda _: all(os.path.isfile(p) for p in needed)).collect()
if not all(visible):
    raise FileNotFoundError("Driver/worker shared-file preflight failed; fix cluster mounts")
print("Workers with required shared files:", sum(visible), "/", len(visible))
print("Actual source bytes:", sum(item["bytes"] for item in source_files))
print("Original January schema:")
spark.read.parquet(paths[0]).printSchema()
raw, source_schemas = load_raw_2019(spark, cfg)
print("Original field counts by file:",
      [(item["month"], item["original_column_count"]) for item in source_schemas])
print("Canonical 12-month Spark schema:")
raw.printSchema()
raw.select("source_file_month", "tpep_pickup_datetime", "PULocationID",
           "fare_amount", "tip_amount", "payment_type").limit(5).show(truncate=False)


## 3️⃣ DOMAIN FEATURE ENGINEERING + VALID-LABEL FILTER

Use a Spark join to the official taxi-zone lookup; target = `tip_amount / fare_amount > 0.20` **only for trips paid by card with valid observed fare/tip and 2019 pickup**. Cash tips are not recorded. Derive borough, hour, weekday and circular month from pickup. Final RatecodeID, payment type, fare, tip, realised distance and drop-off are NOT model inputs. The preview is a small live query; full counts are computed once by Step 4.

In [ ]:
from coursework.data import (prepare_tlc, load_zones, CATEGORICAL_FEATURES,
    NUMERIC_FEATURES, FORBIDDEN_AT_PICKUP)
zones = load_zones(spark, cfg)
eligible = prepare_tlc(raw, zones)  # lazy Spark transformation
features_at_pickup = set(CATEGORICAL_FEATURES) | set(NUMERIC_FEATURES)
assert features_at_pickup.isdisjoint(FORBIDDEN_AT_PICKUP)
print("Pickup-time predictors:", sorted(features_at_pickup))
print("Excluded late/outcome fields:", sorted(FORBIDDEN_AT_PICKUP))
eligible.select("label", "pickup_borough", "pickup_hour", "pickup_dow",
                "pickup_month_sin", "pickup_month_cos").limit(5).show(truncate=False)


## 4️⃣ PARTITIONING + PARQUET STORAGE (ONE AUTHORITATIVE RUN)

This step runs all full-source quality counts, `repartition(shuffle_partitions, pickup_date, PULocationID)`, and one `write.partitionBy('year_month').parquet(...)` in `coursework/data.py`. The plan inspection is lazy; **do not duplicate the 60M-row Parquet write** in a second cell. The real `task1.json` records row counts, bytes, schema, partition settings and stage names.

In [ ]:
from coursework.data import run_task1, load_processed
planned = eligible.repartition(int(cfg["spark"]["shuffle_partitions"]),
                               "pickup_date", "PULocationID")
print("Requested hash partitioning:", int(cfg["spark"]["shuffle_partitions"]),
      "over pickup_date + pickup zone (not a single skewed borough)")
planned.explain()  # no second full write or second full count
observed_1 = run_task1(spark, cfg)  # full 12-file Spark scan and Parquet write, once
processed = load_processed(spark, cfg)
print("Raw rows / retained eligible card trips:",
      observed_1["raw_row_count"], observed_1["clean_row_count"])
print("Recorded before/after partition counts:",
      observed_1["partition_count_before"], observed_1["partition_count_after_repartition"])
print("Partitioned Parquet location:", observed_1["processed_parquet_path"])
processed.printSchema()


## 5️⃣ CACHING STRATEGY (BOUNDED AGGREGATES)

Unlike the reference's unconditional full-data `persist()`, stage Parquet on shared storage and cache only a **small reused monthly aggregate** here. The first action fills it; the second reuses it; unpersist when done. Task 3 separately benchmarks whether caching the training data pays off **including cache-fill cost**.

In [ ]:
from pyspark.storagelevel import StorageLevel
monthly_grouped = processed.groupBy("year_month", "label").count().persist(
    StorageLevel.MEMORY_AND_DISK)
try:
    print("Cached month × label aggregate groups:", monthly_grouped.count())
    monthly_pd = monthly_grouped.orderBy("year_month", "label").toPandas()  # <=24 rows
finally:
    monthly_grouped.unpersist(blocking=True)
from IPython.display import display
display(monthly_pd)


## 6️⃣ VECTOR ASSEMBLY + FOLD-SAFE PREPROCESSING

`coursework/models.py` defines fixed pickup-time `StringIndexerModel` vocabularies, plus learned `Imputer`, `OneHotEncoder`, `VectorAssembler` and scaler stages. Task1 fits an illustrative preprocessing-only PipelineModel on <=1,500 **Jan–Sep** rows for EP1; actual Task2 learns a NEW imputer/encoder/scaler **inside every CV fold** and in each full-training refit. Never fit transforms using test rows.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from coursework.models import make_preprocessing_pipeline
prep = make_preprocessing_pipeline()
for i, stage in enumerate(prep.getStages(), 1):
    print(f"Pipeline stage {i}: {type(stage).__name__}")
assembler = next(s for s in prep.getStages() if isinstance(s, VectorAssembler))
assert set(assembler.getInputCols()).isdisjoint(FORBIDDEN_AT_PICKUP)
print("Assembled, TRAIN-FITTED feature inputs:", assembler.getInputCols())
print("EP1 fitted TRAIN-ONLY demonstrator stages:", observed_1["preprocessing_stages"])
print("Task2 independently FITS per CV fold; no preprocessing is fitted on held-out data.")


## 7️⃣ TRAIN / THRESHOLD / FUTURE TEST SPLIT

January–September 2019 is for CV and full final fitting; October tunes each decision threshold; November–December remains untouched until one final evaluation. Do **not** use the reference project's random 80/20 split for time-sensitive taxi data. Counts below are measured in Step 4.

In [ ]:
from coursework.data import split_processed
train, october, final_test = split_processed(processed)  # Spark DataFrames, lazy
splits = observed_1["temporal_split_rows"]
assert sum(splits.values()) == observed_1["clean_row_count"]
print("Measured Jan–Sep / October / Nov–Dec rows:", splits)
print("Fold candidates are calendar-day-grouped within Jan–Sep (not forward-chaining).")


## 8️⃣ DATA QUALITY, SOURCE VOLUME + FIVE Vs

Show actual original-file fields/bytes, monthly rows and **non-exclusive** data-quality conditions. Card-only filtering is an outcome-observation restriction, not just bad-data deletion. Monthly historic totals are NOT a measured streaming ingestion velocity.

In [ ]:
import pandas as pd
from IPython.display import display
month_counts = pd.DataFrame([
    {"source_month": month, "raw_rows": count}
    for month, count in observed_1["raw_rows_by_source_month"].items()
]).sort_values("source_month")
display(month_counts)
quality_table = pd.DataFrame([
    {"condition (overlaps permitted)": name, "rows": count}
    for name, count in observed_1["quality_counts_overlapping"].items()
])
display(quality_table)
print("Original columns / compressed source GiB:", observed_1["raw_column_count"],
      observed_1["file_size_gib"])
print("Mandatory dataset checks:", observed_1["big_data_verification"])


## 9️⃣ EDA PLOTS — MEASURED MONTHLY LABELS + BOROUGH RATES

All heavy groupBy operations are Spark; only at most 24 month-label cells and a few borough aggregates are collected for matplotlib. Counts/rates describe the **eligible card-only cohort**, not a causal effect or a complete census of cash tips. Do not publish trip-level extracts.

In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
monthly_pd["year_month"] = monthly_pd["year_month"].astype(str)
ax = monthly_pd.pivot(index="year_month", columns="label", values="count").fillna(0).plot(
    kind="bar", stacked=True, figsize=(11, 3.5), color=["#315c84", "#dc784d"])
ax.set(title="Eligible card trips: recorded label by pickup month",
       xlabel="2019 pickup month", ylabel="Trips")
plt.tight_layout()
plt.show()
borough_pd = (processed.groupBy("pickup_borough")
              .agg(F.count(F.lit(1)).alias("n"), F.avg("label").alias("high_tip_rate"))
              .where(F.col("n") >= int(cfg["model"]["minimum_borough_size"]))
              .orderBy("pickup_borough").toPandas())  # few aggregate boroughs
print("Boroughs meeting disclosure threshold:")
display(borough_pd)
ax = borough_pd.plot.bar(x="pickup_borough", y="high_tip_rate", legend=False,
                         figsize=(7, 3), color="#168a85")
ax.set(xlabel="Pickup borough", ylabel="Recorded >20% tip rate",
       title="Observed borough rates (card-only cohort; not causal)")
plt.tight_layout()
plt.show()


## 🔟 ONE-CELL EP1 IMAGE + YOUR INTERPRETATION

This SINGLE cell redraws one ordered composite from this run's measured SparkSession, original printSchema(), raw row/column counts, Hadoop file bytes, partition counts and ACTUAL FITTED Jan–Sep-only preprocessing stages. The demonstrator is NOT a full fitted Task2 classifier. Write your own five-V/ethics analysis and verify identity and allocated-source terms before committing.

In [ ]:
from IPython.display import Image, display
from coursework.evidence import task1_evidence_pack
assert observed_1["status"] == "observed"
assert all(observed_1["big_data_verification"].values())
evidence = project_path(cfg, "results_dir") / "task1_evidence.png"
task1_evidence_pack(observed_1, evidence)  # one cell -> one EP1 composite image
display(Image(filename=str(evidence)))
print("Measured provenance/result:", project_path(cfg, "results_dir") / "task1.json")
print("To explain in YOUR report: 5 Vs, overlap in quality flags, card-only population,")
print("monthly seasonal patterns, why final outcome fields are not model features.")


**Task 1 completion:** review the actual output and report caveats, commit your genuinely observed progress to the required **private module-organisation** repository, then move to Task 2. No execution output or sample score is supplied in this template.